In [16]:
import csv
import numpy as np
from tqdm import tqdm
import ast

In [17]:
# Initialize lists to store the columns
column1 = []
column2 = []

# Read the CSV file
with open("data2D.csv", mode="r", newline="") as file:
    reader = csv.reader(file)
    for row in reader:
        if row:  # Check if the row is not empty
            column1.append(np.array(ast.literal_eval(row[0])))  # First column
            column2.append(int(row[1]))  # Second column

In [22]:
X = np.array(column1).reshape(958, 9, 1)
Y = np.array(column2).reshape(958, 1, 1)

In [33]:
train_X = X[:650]
val_X = X[650:750]
test_X = X[750:]
train_Y = Y[:650]
val_Y = Y[650:750]
test_Y = X[750:]

In [40]:
train_X.shape, train_Y.shape

((650, 9, 1), (650, 1, 1))

In [51]:
class NeuralNetwork:

    def __init__(self, X, Y):

        W0 = np.random.normal(loc=0, scale=np.sqrt(4 / (9 + 100)), size=(1, 100, 9))

        B0 = np.random.normal(loc=0, scale=0, size=(1, 100, 1))

        W1 = np.random.normal(loc=0, scale=np.sqrt(4 / (100 + 100)), size=(1, 100, 100))

        B1 = np.random.normal(loc=0, scale=0, size=(1, 100, 1))

        W2 = np.random.normal(loc=0, scale=np.sqrt(4 / (1 + 100)), size=(1, 1, 100))

        B2 = np.random.normal(loc=0, scale=0, size=(1, 1, 1))

        self.W = [W0, W1, W2]

        self.B = [B0, B1, B2]

        self.X = X

        self.Y = Y

        self.f = [0, 0, 0]

        self.h = [X, 0, 0]

        self.gradients_f = [0, 0, 0]

        self.gradients_W = [0, 0, 0]

        self.gradients_B = [0, 0, 0]

        self.momentum_B = [0, 0, 0]
        self.momentum_W = [0, 0, 0]

        self.beta = 0.9
        # self.losses = []

    def forward_pass(self):

        for i in range(2):

            self.f[i] = self.W[i] @ self.h[i] + self.B[i]

            self.h[i + 1] = np.maximum(self.f[i], 0)

        self.f[2] = self.W[2] @ self.h[2] + self.B[2]

    def get_gradients(self):

        diagonal_matrices_1 = []
        for i in range(len(X)):
            diagonal_matrix_1 = np.diag(self.f[1][i].flatten())
            np.fill_diagonal(
                diagonal_matrix_1, np.maximum(diagonal_matrix_1.diagonal(), 0)
            )

            diagonal_matrices_1.append(diagonal_matrix_1)

        diagonal_matrices_1 = np.array(diagonal_matrices_1)

        diagonal_matrices_0 = []
        for i in range(len(X)):
            diagonal_matrix_0 = np.diag(self.f[0][i].flatten())
            np.fill_diagonal(
                diagonal_matrix_0, np.maximum(diagonal_matrix_0.diagonal(), 0)
            )

            diagonal_matrices_0.append(diagonal_matrix_0)

        diagonal_matrices_0 = np.array(diagonal_matrices_0)

        self.gradients_f[2] = 2 * (self.f[2] - self.Y)
        self.gradients_f[1] = self.gradients_f[2] @ self.W[2] @ diagonal_matrices_1
        self.gradients_f[0] = self.gradients_f[1] @ self.W[1] @ diagonal_matrices_0

        for i in range(3):
            self.gradients_B[i] = np.mean(
                np.transpose(self.gradients_f[i], (0, 2, 1)), axis=0
            )[np.newaxis, :, :]

            self.gradients_W[i] = np.mean(
                np.transpose(self.h[i] @ self.gradients_f[i], (0, 2, 1)), axis=0
            )[np.newaxis, :, :]

    def gradient_descent(self, learning_rate=0.00000001):

        for i in range(3):

            self.momentum_B[i] = (
                self.beta * self.momentum_B[i] + (1 - self.beta) * self.gradients_B[i]
            )
            self.momentum_W[i] = (
                self.beta * self.momentum_W[i] + (1 - self.beta) * self.gradients_W[i]
            )

            self.B[i] = self.B[i] - learning_rate * self.momentum_B[i]
            self.W[i] = self.W[i] - learning_rate * self.momentum_W[i]

    def calculate_mse(self):
        return np.mean((self.Y - self.f[2]) ** 2)


    def train(self, epochs=100, learning_rate=0.0000001):

        for epoch in tqdm(range(epochs)):


            self.forward_pass()
            self.get_gradients()
            self.gradient_descent(learning_rate)

            # self.losses.append(self.calculate_mse())

In [55]:
nn = NeuralNetwork(train_X, train_Y)
nn.beta = 0.5

In [56]:
nn.forward_pass()

In [58]:
nn.f[1][1].shape

(100, 1)

In [59]:
nn.get_gradients()

IndexError: index 650 is out of bounds for axis 0 with size 650